# Frontier: feasibility-tiered roadmap to a full formal proof of MLC

This notebook records the **actual** current remaining frontier of the repository
and an honest, tier-by-tier feasibility assessment of the path to a fully
axiom-free formal proof of the Mandelbrot Local Connectivity conjecture
(`MLC.mlc_conjecture`).

It reflects only the present checked state; superseded intermediate reductions are
not retained here.

## Current checked state

`make check` reports that `MLC.mlc_conjecture` is **`sorry`-free** and depends on
exactly three Lean-core axioms plus **two** non-core project axioms:

```
The proof of 'MLC.mlc_conjecture' is free of 'sorry'.
Core:  Quot.sound, propext, Classical.choice
Project:
  A. MLC.green_sublevel_translate_inter_mandelbrot_connected
  B. MLC.residualOpenVirtualNearMoleculeAxiom
```

`make build`, `make check`, and `scripts/verify_output.sh` all pass.

A full formal proof of MLC means: replace axioms **A** and **B** by theorems
(introducing no new axioms), leaving only the three Lean-core axioms.

## Discharged inputs (now theorems)

The following earlier frontier axioms are **no longer on the checked frontier**;
they are genuine theorems in the repository and are recorded here only to fix
scope:

- **`filled_julia_set_connected`** — connectivity of `K(c)` for `c ∈ M`, now
  `filled_julia_set_connected_proved` (`Mlc/FilledJuliaConnected.lean`).
- **`external_ray_map_exists`** — external-ray / inverse-Böttcher existence for
  `c ∈ M`, discharged via the Green-ray route.
- **`extended_ray_map_free_continuous`** and
  **`green_function_strictMono_along_ray_basin_seam`** — the two unsound ray-seam
  axioms. Both are removed by **Route A**: the Green sublevel sets
  `{z | G_c(z) < ε}` are connected for `c ∈ M` directly by potential theory
  (`green_sublevel_connected_direct`), because `G_c` is harmonic on the basin of
  infinity (`green_function_harmonicOnNhd_basin`) and a harmonic minimum
  principle forces every connected component of a sublevel set to meet `K(c)`.
  The former route through `green_sublevel_joined_to_Kc` (which carried these
  axioms) is no longer used.

## How the proof is currently reduced

The checked architecture reduces MLC to the two remaining axioms through this
chain:

- **Local connectivity of `M`** is obtained from **para-puzzle connectivity**:
  `ParaPuzzlePieceAt c n ∩ M` is connected for `c ∈ M`, all `n`
  (`para_puzzle_piece_inter_mandelbrot_connected_proved`).
- Para-puzzle connectivity is assembled from
  - **dynamical-plane Green-sublevel connectivity** for `c ∈ M`, now a theorem
    (`green_sublevel_connected_onM`, via Route A); this shows each Green sublevel
    `{G_c < (1/2)^n}` is connected, hence equals the dynamical puzzle piece
    `D_n(0)`, and translating to parameter space gives
    `ParaPuzzlePieceAt c n = {c' | G_c(c'-c) < (1/2)^n}`; and
  - a **parameter-plane connectivity** input
    (`green_sublevel_translate_inter_mandelbrot_connected`) — axiom **A**.
- The **infinite-branch / renormalization** part of the classification bottoms
  out in the single residual research axiom **B**
  (`residualOpenVirtualNearMoleculeAxiom`).

So the frontier now splits cleanly into **one parameter-plane axiom of
known-in-principle mathematics** (A) and **one genuinely open research axiom**
(B).

## The two axioms, exact statements

**A. `green_sublevel_translate_inter_mandelbrot_connected`**

```lean
axiom green_sublevel_translate_inter_mandelbrot_connected (c : ℂ)
    (hc : c ∈ MandelbrotSet) (n : ℕ) :
    IsConnected ({c' | green_function c (c' - c) < (1 / 2 : ℝ) ^ n} ∩ MandelbrotSet)
```

Parameter-puzzle connectivity: for every `c ∈ M` and depth `n`, the translated
Green sublevel set intersected with `M` is connected (holomorphic-motion /
λ-lemma content).

**B. `residualOpenVirtualNearMoleculeAxiom`**

```lean
axiom residualOpenVirtualNearMoleculeAxiom :
  ResidualOpenVirtualNearMoleculeData
```

Defined as `Problem43PseudoSiegelAPrioriBoundsData ∧ Problem44VirtualMoleculeData`:
pseudo-Siegel a priori bounds in the remaining **unbounded satellite** ql cases
(Problem 4.3) together with the **virtual Molecule** near-degenerate
interpolation regime (Problem 4.4).

## Feasibility tiers

| Axiom | Content | Tier | Feasibility |
|---|---|---|---|
| A. parameter-puzzle connectivity | λ-lemma / holomorphic motions | **C** | Low-medium — known math, large Mathlib gap |
| B. virtual near-Molecule residual | Dudko-Lyubich open program | **D** | Not currently feasible — open research |

**Tier C** — known mathematics but requiring large missing foundations (or only
available on a sub-stratum of parameters).  
**Tier D** — the genuine open research frontier; formalization is blocked on the
underlying mathematics not being complete/published.

## Tier C — axiom A (parameter-puzzle connectivity)

`{c' | G_c(c'-c) < (1/2)^n} ∩ M` connected is the parameter-plane Yoccoz puzzle
connectivity: transport the (now-proven, via Route A) *dynamical* piece
connectivity to parameter space through a **holomorphic motion of the puzzle
boundary + the λ-lemma (Mañé–Sad–Sullivan / Słodkowski)**.

### λ-lemma foundation (in progress)

`Mlc/Quadratic/Complex/Bottcher/LambdaLemma.lean` builds this genuinely, all
sorry-free and axiom-clean:

- **Analytic core** — `crossTrack_mem_compl`: for three distinct points of a
  motion `E`, the normalized trajectory
  `t ↦ (f t z - f t u)/(f t w - f t u)` lands in `ℂ \ {0,1}`; plus
  `differentiableOn_crossTrack` (holomorphic in time).
- **Disk half of the estimate** — `track_dist_le_of_mapsTo`: the Schwarz–Pick
  Lipschitz bound on a single motion track (via Mathlib's disk Schwarz lemma).
- **Connectivity transport** — `isConnected_image` / `isPreconnected_image`: a
  motion sends a connected set to a connected image, *given* continuity in `z`.

### Sharp residual

The one missing step is **continuity in `z`** (`ContinuousOn (f t) E`), i.e. the
Mañé–Sad–Sullivan continuity theorem. It consumes `crossTrack_mem_compl`
together with the **hyperbolic contraction of `ℂ \ {0,1}` (Schottky /
Montel–Carathéodory)** — equivalently the universal cover `ℍ → ℂ \ {0,1}` (the
modular λ-function). **Mathlib has none of this** (no Schottky, Picard,
thrice-punctured hyperbolic metric, or modular λ), while it *does* have the unit-
disk Schwarz–Pick lemma. So the remaining foundation to close axiom A is
building the hyperbolic metric of `ℂ \ {0,1}` / Schottky's theorem — a large,
self-contained analytic development. **Feasibility: low-medium; the residual is
now sharply isolated as a single classical theorem absent from Mathlib.**

## Tier D — axiom B, the genuine open frontier

`residualOpenVirtualNearMoleculeAxiom` packages

- **Problem 4.3** — pseudo-Siegel a priori bounds in the remaining *unbounded
  satellite* ql cases (`MoleculeImpliesUniformConformalLowerBoundTarget`), and
- **Problem 4.4** — the *virtual Molecule* near-degenerate interpolation regime
  (`IRNoTowerImpliesPrimitiveData`).

These are precisely the currently **open** pieces of the Dudko-Lyubich
'near-Molecule renormalization' program (the §4.3-§4.5 problems of the reference
paper). Unlike Tier C, this is not merely an unformalized-but-known result: the
mathematics itself is not complete in the literature. **A fully axiom-free formal
proof of MLC is therefore blocked here**, and will remain so until the underlying
program is completed mathematically. Formalization cannot get ahead of the
mathematics on this axiom.

The correct posture is to keep axiom B as the single, clearly-labelled residual,
and to ensure the *rest* of the scaffold (now down to axiom A) is discharged so
that the formal development isolates exactly the open mathematical content.

## Recommended path (most feasible ordering)

1. **Discharge axiom A** (parameter-puzzle connectivity): build (or import) the
   λ-lemma / holomorphic-motion foundation and transport the now-proven
   dynamical Green-sublevel connectivity (Route A) to parameter space; or restrict
   to the Yoccoz-accessible finitely-renormalizable stratum and prove it there,
   routing the residual stratum through axiom B.
2. **Leave axiom B** as the labelled open residual. Track it against the
   Dudko-Lyubich Problems 4.3 / 4.4 and only formalize once the mathematics is
   available.

### Bottom line on feasibility

- Axiom **A** is known mathematics but requires a substantial Mathlib-level
  foundation (holomorphic motions); it is the highest-value near-term target now
  that the dynamical-plane connectivity it consumes is fully proven.
- Axiom **B** is the true open frontier; a complete formal MLC proof is not
  currently attainable because the underlying mathematics is itself open.